# RAG Minecraft

## Configuration Gemini API

In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import format_document
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
from langchain_core.stores import InMemoryByteStore
from langchain_classic.retrievers import MultiVectorRetriever
import pickle
import shutil
import os
import csv
import requests
from urllib.parse import unquote
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown
import cloudscraper
import time

C:\Users\chahi\AppData\Local\Temp\ipykernel_21784\3770355338.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

## Scrapping

Congig sources

In [3]:
WIKI_PAGES = [
    "Minecraft"
]

FANDOM_URLS = [
    "https://minecraft.fandom.com/fr/wiki/Survie",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif",
    "https://minecraft.fandom.com/fr/wiki/Hardcore",
    "https://minecraft.fandom.com/fr/wiki/Fabrication",
    "https://minecraft.fandom.com/fr/wiki/Cuisson",
    "https://minecraft.fandom.com/fr/wiki/Alchimie",
    "https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures"
]

In [4]:
scraper = cloudscraper.create_scraper()

In [5]:
def write_csv(file_name, paragraphs):
    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

In [6]:
def write_csv(file_name, paragraphs):
    # Create parent folder automatically if it does not exist
    folder = os.path.dirname(file_name)
    if folder:
        os.makedirs(folder, exist_ok=True)

    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

Wikipedia

In [7]:
def scrape_wikipedia(title: str):

    url = "https://fr.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "page": title,
        "format": "json",
        "prop": "text",
        "redirects": True
    }

    r = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"})
    data = r.json()

    html = data["parse"]["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")

    texts = []
    capture = False

    for tag in soup.find_all(["h2", "p"]):

        t = tag.get_text(" ", strip=True)

        if tag.name == "h2" and "Références" in t:
            break

        if tag.name == "h2":
            capture = True
            continue

        if capture and t:
            texts.append(t)

    write_csv("files/wikipedia.csv", texts)

    return Document(
        page_content="\n\n".join(texts),
        metadata={"source": f"wikipedia:{title}"}
    )

Fandom

In [8]:
def scrape_fandom(url: str):

    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "fr-FR,fr;q=0.9",
    "Referer": "https://www.google.com/"
    }

    r = scraper.get(url, headers=headers)

    soup = BeautifulSoup(r.text, "html.parser")

    content = soup.find("div", {"class": "mw-parser-output"})
    if not content:
        return None

    texts = []

    for tag in content.find_all(["p", "h2", "h3", "li"]):

        t = tag.get_text(" ", strip=True)

        if not t:
            continue

        if len(t) < 20:
            continue

        if "Voir aussi" in t or "Notes et références" in t:
            break

        texts.append(t)

    paragraphs = [p.strip() for p in texts if p.strip()]
    page_name = url.split("/wiki/")[-1]
    page_name = unquote(page_name)
    file_name = f"files/{page_name}.csv"

    write_csv(file_name, paragraphs)


    return Document(
        page_content="\n\n".join(texts),
        metadata={"source": url}
    )

Build dataset

In [9]:
docs = []

# Wikipedia
for page in WIKI_PAGES:
    try:
        docs.append(scrape_wikipedia(page))
        print("WIKI OK:", page)
    except Exception as e:
        print("WIKI ERROR:", page, e)

# Fandom
for url in FANDOM_URLS:
    try:
        doc = scrape_fandom(url)
        if doc:
            docs.append(doc)
            print("FANDOM OK:", url)
    except Exception as e:
        print("FANDOM ERROR:", url, e)

print("\nTOTAL DOCS:", len(docs))

WIKI OK: Minecraft
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cuisson
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures

TOTAL DOCS: 7


Chunking

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=150
)

split_docs = splitter.split_documents(docs)

print("CHUNKS:", len(split_docs))

CHUNKS: 84


Enrchich

In [11]:
def enrich_with_source(docs):
    enriched = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        source_type = "WIKIPEDIA" if "wikipedia" in src else "FANDOM"

        # Extraire le nom de la page depuis l'URL comme "pseudo-titre"
        if "wiki/" in src:
            page_title = src.split("wiki/")[-1].replace("_", " ")
        elif "wikipedia:" in src:
            page_title = src.split("wikipedia:")[-1].replace("_", " ")
        else:
            page_title = src

        # Enrichissement du chunk : on ajoute le contexte AVANT le texte
        # → la similarité vectorielle bénéficie du titre + type de source
        d.page_content = (
            f"SOURCE: {src}\n"
            f"SOURCE_TYPE: {source_type}\n"
            f"TOPIC: {page_title}\n\n"   # ← nouveauté
            f"{d.page_content}"
        )
        enriched.append(d)
    return enriched

split_docs = enrich_with_source(split_docs)

## Initialize embedding model + Retriever

In [12]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

#### Store data using chroma

In [ ]:
import shutil
import os
path = "./chroma_minecraft_db"

if os.path.exists(path):
    try:
        shutil.rmtree(path)
    except PermissionError:
        print("dossier encore utilisé → restart kernel puis relance")

In [ ]:
# --- ÉTAPE 1 : Générer un résumé par chunk ---
# On demande au LLM de résumer chaque chunk en 3 phrases.
# C'est CE résumé qui sera embedé et stocké dans Chroma.
# Pourquoi ? Parce que "C'est quoi le Warden ?" matche mieux
# sur "Le Warden est un mob hostile des profondeurs..." 
# que sur un chunk brut plein de balises et de bruit.

summarize_chain = (
    PromptTemplate.from_template(
        "Résume ce passage Minecraft en 3 phrases claires:\n\n{doc}"
    )
    | llm
    | StrOutputParser()
)

# On génère tous les résumés (attention au rate limit Gemini)
summaries = []
for doc in split_docs:
    summary = summarize_chain.invoke({"doc": doc.page_content})
    summaries.append(summary)
    time.sleep(2)  # rate limiting Gemini

# --- ÉTAPE 2 : Associer chaque résumé à son chunk original ---
# On crée un ID unique par chunk pour faire le lien
# résumé (dans Chroma) <-> chunk original (dans le docstore)
import uuid
doc_ids = [str(uuid.uuid4()) for _ in split_docs]

# Les résumés sont les documents indexés dans Chroma
# On colle l'ID dans les metadata pour retrouver l'original
summary_docs = [
    Document(
        page_content=summaries[i],
        metadata={"doc_id": doc_ids[i]}  # clé de liaison
    )
    for i in range(len(split_docs))
]

# --- ÉTAPE 3 : Construire le retriever ---
# InMemoryByteStore = le "tiroir" qui stocke les chunks originaux
# Le MultiVectorRetriever fait le pont :
#   1. cherche dans Chroma (résumés)
#   2. récupère l'ID du résumé trouvé
#   3. va chercher le chunk original dans le docstore
#   4. retourne le chunk original au LLM


vectorstore = Chroma(
    persist_directory="./chroma_minecraft_multivec",
    embedding_function=gemini_embeddings
)
vectorstore.add_documents(summary_docs)

store = InMemoryByteStore()  # stockage des chunks originaux en mémoire

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,   # Chroma contient les résumés
    byte_store=store,          # le store contient les chunks bruts
    id_key="doc_id",           # la clé qui fait le lien
)

# On peuple le docstore avec les chunks originaux
retriever.docstore.mset(list(zip(doc_ids, split_docs)))

C:\Users\chahi\AppData\Local\Temp\ipykernel_21784\2364975380.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [16]:
# Check the number of chunks generated after text splitting.
print("split_docs:", len(split_docs))

# Check the number of documents actually stored in the Chroma vector database.
# If this number is equal to len(split_docs), the database was built without missing or duplicated chunks.
print("chroma:", vectorstore._collection.count())

split_docs: 84
chroma: 84


#### Create a retriever using Chroma

In [17]:
print(retriever.invoke("Minecraft")[0].page_content[:500])

def retrieve_documents(retriever, query):
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise AttributeError("Le retriever ne supporte ni invoke ni get_relevant_documents.")

SOURCE: https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon
SOURCE_TYPE: FANDOM
TOPIC: Tutoriels/L%27End et l%27Ender Dragon

Exploiter les créatures hostiles

Exploiter les générateurs de monstres

Cultiver des champignons

Culture de disques de musique

Cultiver l'obsidienne

Puits de mine vertical avec eau

Jardin à la française

Motifs en terre cuite émaillée

Mécanismes électroniques avancés

Circuits de redstone simples

Télégraphe en redstone

Comment installer une 


## Generator

#### Create prompt templates

In [18]:
# Prompt template （more strict） to query Gemini
llm_prompt_template = """Tu es un assistant expert sur le jeu Minecraft.
Réponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.
Si la réponse ne se trouve pas dans le contexte ou si tu n'es pas sûr, n'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l'information n'est pas dans les documents fournis."
Sois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l'information.

Question: {question}
Contexte: {context}
Réponse:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template='Tu es un assistant expert sur le jeu Minecraft.\nRéponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.\nSi la réponse ne se trouve pas dans le contexte ou si tu n\'es pas sûr, n\'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l\'information n\'est pas dans les documents fournis."\nSois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l\'information.\n\nQuestion: {question}\nContexte: {context}\nRéponse:'


#### Create a stuff documents chain

In [ ]:
# Combine data from documents to readable string format.
def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content

        formatted_text = f"[{source}]\n{content}"

        formatted_docs.append(formatted_text)

    return "\n\n".join(formatted_docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### RAG AVANCÉ : ACTIVE RETRIEVAL (RRR)

In [ ]:
def check_if_answered(reponse):
    prompt = f"La réponse suivante indique-t-elle qu'elle ne trouve pas l'information ou qu'elle ne sait pas ? Réponds OUI ou NON.\nRéponse : {reponse}"
    result = llm.invoke(prompt)
    text_content = str(result.content)
    return "OUI" in text_content.strip().upper()

def ask_with_active_retrieval(question):
    print(f"▶ Question posée : {question}")
    
    reponse = rag_chain.invoke(question)
    phrase_echec = "Je suis désolé, mais l'information n'est pas dans les documents fournis." # Phrase exite dans llm_prompt_template，à détecter pour déclencher l'auto-correction
    
    # Auto-correction
    if check_if_answered(reponse):
        print("Information introuvable. Activation de l'Active Retrieval...")
        
        # 1. IA de réécriture (Query Optimizer)
        llm_rewrite = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
        rewrite_prompt = f"""Tu es un expert Minecraft. Un utilisateur a posé cette question : '{question}'. 
        Cette question n'a donné aucun résultat exact dans notre base de données RAG. 
        Réécris cette question sous forme de 2 ou 3 mots-clés très simples et percutants pour optimiser une recherche par similarité vectorielle. 
        Ne renvoie QUE les mots-clés (par exemple : 'Ender Dragon stratégie'), rien d'autre."""
        
        # Générer la nouvelle requête
        nouvelle_requete = str(llm_rewrite.invoke(rewrite_prompt).content)
        print(f"Nouvelle requête optimisée par l'IA : {nouvelle_requete}")
        
        # 2. Relancer la recherche avec les nouveaux mots-clés
        reponse_niveau_2 = rag_chain.invoke(nouvelle_requete)
        
        # 3. Vérifier si la deuxième tentative a réussi
        if phrase_echec not in reponse_niveau_2:
            print("Succès du Niveau 2.")
            return f"*(Auto-correction RRR : Recherche élargie avec les mots-clés '{nouvelle_requete}')*\n\n{reponse_niveau_2}"
        else:
            print("Échec du Niveau 2. L'information n'existe définitivement pas dans la base.")
            return reponse # Renvoie le message de refus standard
            
    # Si le niveau 1 a marché directement
    print("Succès du Niveau 1.")
    return reponse

### Prompt the model

In [ ]:
Markdown(ask_with_active_retrieval("Quel est l'ingrédient de base indispensable pour l'alchimie ?"))

▶ Question posée : Quel est l'ingrédient de base indispensable pour l'alchimie ?


Information introuvable. Activation de l'Active Retrieval...


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 28.186184702s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi le Warden dans Minecraft ?"))

▶ Question posée : C'est quoi le Warden dans Minecraft ?
Succès du Niveau 1.


Le Warden est un monstre, parfois confondu avec un boss, qui vit dans une « cité antique » située dans les grottes les plus basses du monde. Apparu en 2022 avec la mise à jour 1.19, il peut infliger de très lourds dégâts. À sa mort, il laisse un bloc appelé « catalyseur de sculk ».

[WIKIPEDIA: wikipedia:Minecraft]

In [ ]:
Markdown(ask_with_active_retrieval("Comment survivre dans le Nether dans Minecraft ?"))

▶ Question posée : Comment survivre dans le Nether dans Minecraft ?
Succès du Niveau 1.


Pour survivre dans le Nether, il est essentiel de construire un abri solide autour de votre portail en utilisant des matériaux résistants aux boules de feu des Ghasts, comme de la pierre. Emportez toujours un briquet pour rallumer votre portail si nécessaire, ainsi qu'un établi et des outils de rechange. De plus, marquez votre chemin avec des torches ou des blocs visibles pour éviter de vous perdre dans ce monde dangereux. Enfin, ne posez jamais de lit car il explosera si vous tentez d'y dormir, et sachez que vous pouvez tenter de renvoyer les projectiles des Ghasts en effectuant un clic gauche dessus.

[FANDOM: https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether]

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi l'alchimie dans Minecraft?"))

▶ Question posée : C'est quoi l'alchimie dans Minecraft?
Succès du Niveau 1.


L'alchimie (ou distillation) est la science permettant de créer des potions, des potions jetables et des potions persistantes dans Minecraft. Pour la pratiquer, le joueur doit utiliser un alambic alimenté par de la poudre de Blaze afin de distiller divers ingrédients dans des fioles de liquide. La verrue du Nether sert généralement de point de départ pour préparer une "potion étrange", à laquelle on ajoute ensuite d'autres ingrédients pour obtenir des effets spécifiques. Enfin, l'utilisation de poudre à canon ou de souffle de dragon permet de transformer ces potions pour les rendre jetables ou persistantes. [FANDOM: https://minecraft.fandom.com/fr/wiki/Alchimie]

In [ ]:
Markdown(ask_with_active_retrieval("Il y a combien de dimensions dans Minecraft? Décris les."))

▶ Question posée : Il y a combien de dimensions dans Minecraft? Décris les.


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 7.812978507s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

In [ ]:
Markdown(ask_with_active_retrieval("Parle moi des boss dans Minecraft, et décris les tous."))

In [ ]:
### ÉVALUATION DU RAG (LLM-as-a-Judge)